# FTC Open Analytics Dataset — Exploration Notebook

This notebook provides an interactive exploration of the FTC Open Analytics Dataset, including match score distributions, team performance metrics, and seasonal trends.

**Dataset**: 877 matches across 6 seasons (1819–2324), 531 unique teams.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 100

# Load data
matches = pd.read_csv("data/processed/matches.csv")
teams = pd.read_csv("data/processed/teams.csv")
team_events = pd.read_csv("data/processed/team_events.csv")

print(f"Loaded: {len(matches)} matches, {len(teams)} teams, {len(team_events)} team-event records")

## 1. Dataset Overview

In [ ]:
print("=== MATCHES ===")
matches.info()
print("\n=== TEAMS ===")
teams.info()
print("\n=== TEAM EVENTS ===")
team_events.info()

## 2. Matches Per Season

In [ ]:
season_counts = matches["season"].value_counts().sort_index()
season_labels = {
    "1819": "Rover Ruckus", "1920": "Skystone", "2021": "Ultimate Goal",
    "2122": "Freight Frenzy", "2223": "Power Play", "2324": "Centerstage"
}

fig, ax = plt.subplots()
bars = ax.bar(
    [season_labels.get(s, s) for s in season_counts.index],
    season_counts.values,
    color=sns.color_palette("viridis", len(season_counts))
)
ax.set_title("Matches Per Season", fontsize=14, fontweight="bold")
ax.set_ylabel("Number of Matches")
for bar, count in zip(bars, season_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, str(count),
            ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

## 3. Score Distribution

In [ ]:
all_scores = pd.concat([matches["red_score"], matches["blue_score"]])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Histogram
axes[0].hist(all_scores, bins=40, color="steelblue", edgecolor="white", alpha=0.8)
axes[0].axvline(all_scores.mean(), color="red", linestyle="--", label=f"Mean: {all_scores.mean():.1f}")
axes[0].axvline(all_scores.median(), color="orange", linestyle="--", label=f"Median: {all_scores.median():.1f}")
axes[0].set_title("Score Distribution (All Matches)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Score")
axes[0].set_ylabel("Frequency")
axes[0].legend()

# Box plot by season
season_scores = {}
for season in sorted(matches["season"].unique()):
    s_df = matches[matches["season"] == season]
    season_scores[season_labels.get(season, season)] = pd.concat([s_df["red_score"], s_df["blue_score"]]).values

axes[1].boxplot(season_scores.values(), labels=season_scores.keys(), patch_artist=True)
axes[1].set_title("Score Distribution by Season", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Score")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 4. Winner Distribution

In [ ]:
winner_counts = matches["winner"].value_counts()
colors = {"red": "#e74c3c", "blue": "#3498db", "tie": "#95a5a6"}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
axes[0].pie(
    winner_counts.values,
    labels=[w.capitalize() for w in winner_counts.index],
    colors=[colors.get(w, "gray") for w in winner_counts.index],
    autopct="%1.1f%%",
    startangle=90,
    explode=[0.02]*len(winner_counts)
)
axes[0].set_title("Overall Winner Distribution", fontsize=13, fontweight="bold")

# By season
winner_by_season = pd.crosstab(matches["season"], matches["winner"])
winner_by_season.index = [season_labels.get(s, s) for s in winner_by_season.index]
winner_by_season.plot(kind="bar", stacked=True, ax=axes[1], color=[colors.get(c, "gray") for c in winner_by_season.columns])
axes[1].set_title("Winner Distribution by Season", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Number of Matches")
axes[1].legend(title="Winner")
plt.tight_layout()
plt.show()

## 5. OPR Distribution

In [ ]:
valid_oprs = team_events["opr"].dropna()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Histogram
axes[0].hist(valid_oprs, bins=40, color="darkgreen", edgecolor="white", alpha=0.8)
axes[0].axvline(valid_oprs.mean(), color="red", linestyle="--", label=f"Mean: {valid_oprs.mean():.1f}")
axes[0].axvline(0, color="gray", linestyle=":", alpha=0.5, label="Zero")
axes[0].set_title("OPR Distribution", fontsize=13, fontweight="bold")
axes[0].set_xlabel("OPR")
axes[0].set_ylabel("Frequency")
axes[0].legend()

# Top 15 teams by OPR
top_opr = team_events.nlargest(15, "opr")[["team_number", "event_key", "season", "opr", "ccwm"]]
axes[1].barh(
    [f"Team {t} ({s})" for t, s in zip(top_opr["team_number"], top_opr["season"])][::-1],
    top_opr["opr"].values[::-1],
    color="darkgreen", alpha=0.8
)
axes[1].set_title("Top 15 Team-Event Performances (by OPR)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("OPR")
plt.tight_layout()
plt.show()

## 6. OPR vs CCWM Correlation

In [ ]:
valid_team_events = team_events.dropna(subset=["opr", "ccwm"])
corr = valid_team_events["opr"].corr(valid_team_events["ccwm"])

plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=valid_team_events, x="opr", y="ccwm",
    alpha=0.5, edgecolor=None, s=30
)
plt.axhline(0, color="gray", linestyle=":", alpha=0.5)
plt.axvline(0, color="gray", linestyle=":", alpha=0.5)

# Add regression line
m, b = np.polyfit(valid_team_events["opr"], valid_team_events["ccwm"], 1)
x_line = np.linspace(valid_team_events["opr"].min(), valid_team_events["opr"].max(), 100)
plt.plot(x_line, m * x_line + b, color="red", linewidth=2, label=f"r = {corr:.3f}")

plt.title("OPR vs CCWM", fontsize=14, fontweight="bold")
plt.xlabel("Offensive Power Rating (OPR)")
plt.ylabel("Contribution to Winning Margin (CCWM)")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Qualification vs Playoff Scores

In [ ]:
quals_scores = pd.concat([
    matches[~matches["is_playoff"]]["red_score"],
    matches[~matches["is_playoff"]]["blue_score"]
])
playoff_scores = pd.concat([
    matches[matches["is_playoff"]]["red_score"],
    matches[matches["is_playoff"]]["blue_score"]
])

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(quals_scores, bins=30, alpha=0.6, label=f"Qualification (n={len(quals_scores)}, μ={quals_scores.mean():.1f})", color="steelblue")
ax.hist(playoff_scores, bins=30, alpha=0.6, label=f"Playoff (n={len(playoff_scores)}, μ={playoff_scores.mean():.1f})", color="coral")
ax.set_title("Score Distribution: Qualification vs Playoff", fontsize=13, fontweight="bold")
ax.set_xlabel("Score")
ax.set_ylabel("Frequency")
ax.legend()
plt.tight_layout()
plt.show()

## 8. Score Differential Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Histogram of score differentials
axes[0].hist(matches["score_diff"], bins=40, color="purple", edgecolor="white", alpha=0.8)
axes[0].axvline(0, color="gray", linestyle="--", alpha=0.7)
axes[0].set_title("Score Differential Distribution", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Score Differential (Red — Blue)")
axes[0].set_ylabel("Frequency")

# Close matches analysis
close_matches = matches[abs(matches["score_diff"]) <= 10]
blowouts = matches[abs(matches["score_diff"]) >= 80]
print(f"Close matches (|diff| ≤ 10): {len(close_matches)} ({len(close_matches)/len(matches)*100:.1f}%)")
print(f"Blowouts (|diff| ≥ 80): {len(blowouts)} ({len(blowouts)/len(matches)*100:.1f}%)")

# Score differential boxplot by season
season_data = [matches[matches["season"] == s]["score_diff"].values for s in sorted(matches["season"].unique())]
bp = axes[1].boxplot(season_data, labels=[season_labels.get(s, s) for s in sorted(matches["season"].unique())], patch_artist=True)
axes[1].axhline(0, color="gray", linestyle="--", alpha=0.7)
axes[1].set_title("Score Differential by Season", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Score Differential")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 9. Team Performance Summary

In [ ]:
# Top 10 teams by average OPR across all events
team_avg_opr = team_events.groupby("team_number")["opr"].agg(["mean", "count", "max"]).reset_index()
team_avg_opr = team_avg_opr[team_avg_opr["count"] >= 1].sort_values("mean", ascending=False)

print("Top 10 Teams by Average OPR (min 1 event):")
display(team_avg_opr.head(10))

print("\nOPR Summary Statistics:")
print(f"  Mean OPR:  {valid_oprs.mean():.2f}")
print(f"  Median OPR: {valid_oprs.median():.2f}")
print(f"  Std OPR:    {valid_oprs.std():.2f}")
print(f"  Max OPR:    {valid_oprs.max():.2f}")
print(f"  Min OPR:    {valid_oprs.min():.2f}")

## 10. Win Rate vs OPR

In [ ]:
team_events["win_rate"] = team_events["wins"] / (team_events["wins"] + team_events["losses"] + team_events["ties"])
valid_wr = team_events.dropna(subset=["opr", "win_rate"])

corr_wr_opr = valid_wr["win_rate"].corr(valid_wr["opr"])

plt.figure(figsize=(8, 6))
sns.scatterplot(data=valid_wr, x="opr", y="win_rate", alpha=0.4, s=30)
m2, b2 = np.polyfit(valid_wr["opr"], valid_wr["win_rate"], 1)
x_line2 = np.linspace(valid_wr["opr"].min(), valid_wr["opr"].max(), 100)
plt.plot(x_line2, m2 * x_line2 + b2, color="red", linewidth=2, label=f"r = {corr_wr_opr:.3f}")
plt.title("Win Rate vs OPR", fontsize=14, fontweight="bold")
plt.xlabel("Offensive Power Rating (OPR)")
plt.ylabel("Win Rate")
plt.legend()
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrates the key characteristics of the FTC Open Analytics Dataset:

- **877 matches** across 6 seasons with realistic score distributions
- **OPR** is positively correlated with both CCWM and win rate, validating its use as a team strength metric
- **Playoff matches** tend to have slightly higher scores than qualification matches
- Score differentials show a roughly normal distribution centered near zero